In [ ]:

import os
import ast
import re
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("All libraries imported successfully!")
print("Random seed:", SEED)


All libraries imported successfully!
Random seed: 42


In [ ]:

import pandas as pd
from huggingface_hub import hf_hub_download

file_path = hf_hub_download(repo_id="withmartian/routerbench",filename="routerbench_0shot.pkl",
    repo_type="dataset")

df = pd.read_pickle(file_path)

print("RouterBench dataset loaded successfully!")
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nFirst 3 rows:")
print(df.head(3).to_string())


routerbench_0shot.pkl: reconstructing file:   0%|          |  0.00B / 99.6MB            

routerbench_0shot.pkl: downloading bytes:           |  0.00B            

RouterBench dataset loaded successfully!
Dataset shape: (36497, 37)

Column names:
['sample_id', 'prompt', 'eval_name', 'WizardLM/WizardLM-13B-V1.2', 'claude-instant-v1', 'claude-v1', 'claude-v2', 'gpt-3.5-turbo-1106', 'gpt-4-1106-preview', 'meta/code-llama-instruct-34b-chat', 'meta/llama-2-70b-chat', 'mistralai/mistral-7b-chat', 'mistralai/mixtral-8x7b-chat', 'zero-one-ai/Yi-34B-Chat', 'gpt-3.5-turbo-1106|model_response', 'claude-instant-v1|model_response', 'claude-v1|model_response', 'claude-v2|model_response', 'gpt-4-1106-preview|model_response', 'meta/llama-2-70b-chat|model_response', 'mistralai/mixtral-8x7b-chat|model_response', 'zero-one-ai/Yi-34B-Chat|model_response', 'WizardLM/WizardLM-13B-V1.2|model_response', 'meta/code-llama-instruct-34b-chat|model_response', 'mistralai/mistral-7b-chat|model_response', 'gpt-3.5-turbo-1106|total_cost', 'claude-instant-v1|total_cost', 'claude-v1|total_cost', 'claude-v2|total_cost', 'gpt-4-1106-preview|total_cost', 'meta/llama-2-70b-chat|total_

In [ ]:
models = [col for col in df.columns if col.endswith("|total_cost")]
models = [col.replace("|total_cost", "") for col in models]

quality_matrix = df[models].apply(pd.to_numeric, errors="coerce")

cost_matrix = df[[f"{model}|total_cost" for model in models]].apply(pd.to_numeric, errors="coerce")
cost_matrix.columns = models

print("Models:", len(models))
print("\nModel names:\n", "\n".join(models))
print("\nQuality matrix shape:", quality_matrix.shape)
print("Cost matrix shape:", cost_matrix.shape)

Models: 11

Model names:
 gpt-3.5-turbo-1106
claude-instant-v1
claude-v1
claude-v2
gpt-4-1106-preview
meta/llama-2-70b-chat
mistralai/mixtral-8x7b-chat
zero-one-ai/Yi-34B-Chat
WizardLM/WizardLM-13B-V1.2
meta/code-llama-instruct-34b-chat
mistralai/mistral-7b-chat

Quality matrix shape: (36497, 11)
Cost matrix shape: (36497, 11)


In [ ]:

print("Missing values in quality matrix:", quality_matrix.isna().sum().sum())
print("Missing values in cost matrix:", cost_matrix.isna().sum().sum())

print("\nQuality score ranges:")
print(quality_matrix.agg(["min", "max"]).to_string())

print("\nCost ranges:")
print(cost_matrix.agg(["min", "max"]).to_string())

print("\nNegative quality values:", (quality_matrix < 0).sum().sum())
print("Negative cost values:", (cost_matrix < 0).sum().sum())


Missing values in quality matrix: 0
Missing values in cost matrix: 0

Quality score ranges:
     gpt-3.5-turbo-1106  claude-instant-v1  claude-v1  claude-v2  gpt-4-1106-preview  meta/llama-2-70b-chat  mistralai/mixtral-8x7b-chat  zero-one-ai/Yi-34B-Chat  WizardLM/WizardLM-13B-V1.2  meta/code-llama-instruct-34b-chat  mistralai/mistral-7b-chat
min                 0.0                0.0        0.0        0.0                 0.0                    0.0                          0.0                      0.0                         0.0                                0.0                        0.0
max                 1.0                1.0        1.0        1.0                 1.0                    1.0                          1.0                      1.0                         1.0                                1.0                        1.0

Cost ranges:
     gpt-3.5-turbo-1106  claude-instant-v1  claude-v1  claude-v2  gpt-4-1106-preview  meta/llama-2-70b-chat  mistralai/mixtral-8x7b-chat  

In [ ]:

model_summary = pd.DataFrame({ "Average Quality": quality_matrix.mean(),
    "Average Cost": cost_matrix.mean(),
    "Missing Quality": quality_matrix.isna().sum(),
    "Missing Cost": cost_matrix.isna().sum()})

model_summary = model_summary.sort_values("Average Quality", ascending=False)

print(model_summary.to_string())


                                   Average Quality  Average Cost  Missing Quality  Missing Cost
gpt-4-1106-preview                        0.781400      0.003293                0             0
zero-one-ai/Yi-34B-Chat                   0.647492      0.000186                0             0
claude-v2                                 0.635806      0.002419                0             0
claude-v1                                 0.630118      0.002145                0             0
gpt-3.5-turbo-1106                        0.619321      0.000243                0             0
claude-instant-v1                         0.598414      0.000233                0             0
mistralai/mixtral-8x7b-chat               0.547115      0.000135                0             0
WizardLM/WizardLM-13B-V1.2                0.431127      0.000073                0             0
meta/llama-2-70b-chat                     0.328689      0.000203                0             0
mistralai/mistral-7b-chat               

In [ ]:

model_summary["Quality per Cost"] = model_summary["Average Quality"] / model_summary["Average Cost"]

model_summary = model_summary.sort_values("Quality per Cost", ascending=False)

print("Models ranked by quality per unit cost:\n")
print(model_summary.to_string())


Models ranked by quality per unit cost:

                                   Average Quality  Average Cost  Missing Quality  Missing Cost  Quality per Cost
mistralai/mistral-7b-chat                 0.306111      0.000046                0             0       6694.723164
WizardLM/WizardLM-13B-V1.2                0.431127      0.000073                0             0       5912.785526
mistralai/mixtral-8x7b-chat               0.547115      0.000135                0             0       4065.353240
zero-one-ai/Yi-34B-Chat                   0.647492      0.000186                0             0       3490.101972
claude-instant-v1                         0.598414      0.000233                0             0       2568.991773
gpt-3.5-turbo-1106                        0.619321      0.000243                0             0       2545.920587
meta/llama-2-70b-chat                     0.328689      0.000203                0             0       1621.832206
meta/code-llama-instruct-34b-chat         0.202

In [ ]:
strongest_model = quality_matrix.mean().idxmax()
cheapest_model = cost_matrix.mean().idxmin()
best_efficiency_model = (quality_matrix.mean() / cost_matrix.mean()).idxmax()

print("Strongest model:", strongest_model)
print("Average quality:", round(quality_matrix[strongest_model].mean(), 4))
print("Average cost:", round(cost_matrix[strongest_model].mean(), 6))

print("\nCheapest model:", cheapest_model)
print("Average quality:", round(quality_matrix[cheapest_model].mean(), 4))
print("Average cost:", round(cost_matrix[cheapest_model].mean(), 6))

print("\nBest quality-per-cost model:", best_efficiency_model)
print("Average quality:", round(quality_matrix[best_efficiency_model].mean(), 4))
print("Average cost:", round(cost_matrix[best_efficiency_model].mean(), 6))

Strongest model: gpt-4-1106-preview
Average quality: 0.7814
Average cost: 0.003293

Cheapest model: mistralai/mistral-7b-chat
Average quality: 0.3061
Average cost: 4.6e-05

Best quality-per-cost model: mistralai/mistral-7b-chat
Average quality: 0.3061
Average cost: 4.6e-05


In [ ]:
baseline_results = pd.DataFrame({"Average Quality": quality_matrix.mean(),
    "Average Cost Per Prompt": cost_matrix.mean()})

baseline_results["Cost Per 1000 Prompts"] = baseline_results["Average Cost Per Prompt"] * 1000
baseline_results["Quality Retention (%)"] = baseline_results["Average Quality"] / quality_matrix[strongest_model].mean() * 100
baseline_results = baseline_results.sort_values("Average Cost Per Prompt")

print(baseline_results.round(4).to_string())

                                   Average Quality  Average Cost Per Prompt  Cost Per 1000 Prompts  Quality Retention (%)
mistralai/mistral-7b-chat                   0.3061                   0.0000                 0.0457                39.1748
WizardLM/WizardLM-13B-V1.2                  0.4311                   0.0001                 0.0729                55.1737
mistralai/mixtral-8x7b-chat                 0.5471                   0.0001                 0.1346                70.0173
meta/code-llama-instruct-34b-chat           0.2022                   0.0002                 0.1720                25.8802
zero-one-ai/Yi-34B-Chat                     0.6475                   0.0002                 0.1855                82.8630
meta/llama-2-70b-chat                       0.3287                   0.0002                 0.2027                42.0641
claude-instant-v1                           0.5984                   0.0002                 0.2329                76.5822
gpt-3.5-turbo-1106      

In [ ]:
oracle_quality = quality_matrix.max(axis=1)
oracle_model = quality_matrix.idxmax(axis=1)
oracle_cost = cost_matrix.to_numpy()[np.arange(len(df)), quality_matrix.to_numpy().argmax(axis=1)]

print("Oracle average quality:", round(oracle_quality.mean(), 4))
print("Oracle average cost:", round(oracle_cost.mean(), 6))
print("Strongest fixed-model quality:", round(quality_matrix[strongest_model].mean(), 4))
print("Quality improvement over strongest fixed model:", round((oracle_quality.mean() - quality_matrix[strongest_model].mean()) * 100, 2), "percentage points")
print("\nMost frequently oracle-selected models:")
print(oracle_model.value_counts().to_string())

Oracle average quality: 0.9121
Oracle average cost: 0.000596
Strongest fixed-model quality: 0.7814
Quality improvement over strongest fixed model: 13.07 percentage points

Most frequently oracle-selected models:
gpt-3.5-turbo-1106                   23718
claude-instant-v1                     5658
gpt-4-1106-preview                    2314
claude-v1                             2123
claude-v2                             1111
mistralai/mixtral-8x7b-chat            459
zero-one-ai/Yi-34B-Chat                334
WizardLM/WizardLM-13B-V1.2             298
mistralai/mistral-7b-chat              266
meta/llama-2-70b-chat                  120
meta/code-llama-instruct-34b-chat       96


In [ ]:
import ast

def extract_prompt(value):
    if isinstance(value, list):
        return " ".join(str(item) for item in value)
    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return " ".join(str(item) for item in parsed)
        except (ValueError, SyntaxError):
            pass
        return value
    return "" if pd.isna(value) else str(value)

df["clean_prompt"] = df["prompt"].apply(extract_prompt)
df["clean_prompt"] = df["clean_prompt"].str.strip()

print("Total prompts:", len(df))
print("Empty prompts:", df["clean_prompt"].eq("").sum())
print("\nExample cleaned prompts:")
for prompt in df["clean_prompt"].head(3):
    print("\n", prompt[:300])

Total prompts: 36497
Empty prompts: 0

Example cleaned prompts:

 猜字谜，根据我给的描述猜出一个字(请从汉字的字形、发音、意义以及字的拆分组合等角度考虑)。首先提供你的推理，然后提供用英文方括号括[]起来的最终答案。 古字刚分开，豆字挤进来，你先别生气，看后笑开怀。

 猜字谜，根据我给的描述猜出一个字(请从汉字的字形、发音、意义以及字的拆分组合等角度考虑)。首先提供你的推理，然后提供用英文方括号括[]起来的最终答案。 严禁烟火

 猜字谜，根据我给的描述猜出一个字(请从汉字的字形、发音、意义以及字的拆分组合等角度考虑)。首先提供你的推理，然后提供用英文方括号括[]起来的最终答案。 海边无水，种树一棵，开花没叶，会结酸果。


In [ ]:
print("Total prompts:", len(df))
print("Empty prompts:", df["clean_prompt"].eq("").sum())

print("\nSample cleaned prompts:")

for index, prompt in enumerate(df["clean_prompt"].head(3), start=1):
    print(f"\nPrompt {index}:")
    print(prompt[:300])

Total prompts: 36497
Empty prompts: 0

Sample cleaned prompts:

Prompt 1:
猜字谜，根据我给的描述猜出一个字(请从汉字的字形、发音、意义以及字的拆分组合等角度考虑)。首先提供你的推理，然后提供用英文方括号括[]起来的最终答案。 古字刚分开，豆字挤进来，你先别生气，看后笑开怀。

Prompt 2:
猜字谜，根据我给的描述猜出一个字(请从汉字的字形、发音、意义以及字的拆分组合等角度考虑)。首先提供你的推理，然后提供用英文方括号括[]起来的最终答案。 严禁烟火

Prompt 3:
猜字谜，根据我给的描述猜出一个字(请从汉字的字形、发音、意义以及字的拆分组合等角度考虑)。首先提供你的推理，然后提供用英文方括号括[]起来的最终答案。 海边无水，种树一棵，开花没叶，会结酸果。


In [ ]:
!pip -q install langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 25.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
from langdetect import detect, LangDetectException
from collections import Counter

def detect_language(text):
    try:
        return detect(text)
    except LangDetectException:
        return "unknown"

sample_prompts = df["clean_prompt"].sample(n=min(1000, len(df)),
    random_state=42)

language_counts = Counter( detect_language(prompt)
    for prompt in sample_prompts)

print("Language distribution in 1,000 sampled prompts:")

for language, count in language_counts.most_common():
    percentage = count / len(sample_prompts) * 100
    print(f"{language}: {count} ({percentage:.2f}%)")

Language distribution in 1,000 sampled prompts:
en: 988 (98.80%)
zh-cn: 12 (1.20%)


In [ ]:
def is_english(text):
    try:
        return detect(text) == "en"
    except LangDetectException:
        return False

df["language"] = df["clean_prompt"].apply(is_english)

df = df[df["language"]].copy()
df.reset_index(drop=True, inplace=True)

print("English prompts remaining:", len(df))
print("Removed prompts:", 36497 - len(df))
print("\nSample prompts:")
for prompt in df["clean_prompt"].head(3):
    print("\n", prompt[:200])

English prompts remaining: 36178
Removed prompts: 319

Sample prompts:

 please write an title based on this abstract of article:  In this study, the efficient transfer printing and stable operation of the PEDOT:PSS hole extraction layer have been researched in the perovsk

 please write an title based on this abstract of article:  One of the challenges in the field of perovskite solar cells (PSC) is the development of inorganic hole-transporting-materials (HTM) suitable 

 please write an title based on this abstract of article:  Electron transport layers (ETLs) in perovskite solar cells (PSCs) are a key factor to determine the photovoltaic performance. Herein, we demon


In [ ]:
models = [col.replace("|total_cost", "") for col in df.columns if col.endswith("|total_cost")]
quality_matrix = df[models].apply(pd.to_numeric, errors="coerce")
cost_matrix = df[[f"{model}|total_cost" for model in models]].apply(pd.to_numeric, errors="coerce")
cost_matrix.columns = models

print("Dataset:", df.shape)
print("Quality matrix:", quality_matrix.shape)
print("Cost matrix:", cost_matrix.shape)
print("Models:", len(models))
print("Missing quality values:", quality_matrix.isna().sum().sum())
print("Missing cost values:", cost_matrix.isna().sum().sum())

Dataset: (36178, 39)
Quality matrix: (36178, 11)
Cost matrix: (36178, 11)
Models: 11
Missing quality values: 0
Missing cost values: 0


In [ ]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
prompt_embeddings = embedding_model.encode(df["clean_prompt"].tolist(), batch_size=64, show_progress_bar=True, convert_to_numpy=True)

print("Embedding shape:", prompt_embeddings.shape)
print("Expected shape:", (len(df), 384))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/566 [00:00<?, ?it/s]

Embedding shape: (36178, 384)
Expected shape: (36178, 384)


In [ ]:
best_model_indices = quality_matrix.to_numpy().argmax(axis=1)
best_model_labels = np.array(models)[best_model_indices]

print("Number of labels:", len(best_model_labels))
print("\nBest-model distribution:")
print(pd.Series(best_model_labels).value_counts().to_string())
print("\nLabels match prompts:", len(best_model_labels) == len(df))

Number of labels: 36178

Best-model distribution:
gpt-3.5-turbo-1106                   23504
claude-instant-v1                     5652
gpt-4-1106-preview                    2275
claude-v1                             2097
claude-v2                             1098
mistralai/mixtral-8x7b-chat            455
zero-one-ai/Yi-34B-Chat                320
WizardLM/WizardLM-13B-V1.2             298
mistralai/mistral-7b-chat              265
meta/llama-2-70b-chat                  120
meta/code-llama-instruct-34b-chat       94

Labels match prompts: True


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(best_model_labels)

weights = compute_class_weight( class_weight="balanced",
    classes=classes,
    y=best_model_labels)

class_weights = dict(zip(classes, weights))

print("Class weights:")
for model, weight in class_weights.items():
    print(f"{model}: {weight:.3f}")

Class weights:
WizardLM/WizardLM-13B-V1.2: 11.037
claude-instant-v1: 0.582
claude-v1: 1.568
claude-v2: 2.995
gpt-3.5-turbo-1106: 0.140
gpt-4-1106-preview: 1.446
meta/code-llama-instruct-34b-chat: 34.988
meta/llama-2-70b-chat: 27.408
mistralai/mistral-7b-chat: 12.411
mistralai/mixtral-8x7b-chat: 7.228
zero-one-ai/Yi-34B-Chat: 10.278


In [ ]:
from sklearn.model_selection import train_test_split

df = df.reset_index(drop=True)
quality_matrix = quality_matrix.reset_index(drop=True)
cost_matrix = cost_matrix.reset_index(drop=True)

train_idx, test_idx = train_test_split(np.arange(len(df)),
    test_size=0.2,
    random_state=42,
    stratify=best_model_labels)

X_train, X_test = prompt_embeddings[train_idx], prompt_embeddings[test_idx]
y_train, y_test = best_model_labels[train_idx], best_model_labels[test_idx]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("Embedding rows:", len(prompt_embeddings))
print("Quality rows:", len(quality_matrix))
print("Cost rows:", len(cost_matrix))

Training samples: 28942
Testing samples: 7236
Embedding rows: 36178
Quality rows: 36178
Cost rows: 36178


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

router = RandomForestClassifier(n_estimators=200,class_weight=class_weights,
    random_state=42,
    n_jobs=-1)

router.fit(X_train, y_train)

y_pred = router.predict(X_test)

print("Test accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

Test accuracy: 0.6455

Classification report:
                                   precision    recall  f1-score   support

       WizardLM/WizardLM-13B-V1.2       0.00      0.00      0.00        60
                claude-instant-v1       0.05      0.00      0.00      1130
                        claude-v1       0.35      0.01      0.03       419
                        claude-v2       0.00      0.00      0.00       220
               gpt-3.5-turbo-1106       0.65      0.99      0.79      4701
               gpt-4-1106-preview       0.20      0.01      0.01       455
meta/code-llama-instruct-34b-chat       0.00      0.00      0.00        19
            meta/llama-2-70b-chat       0.00      0.00      0.00        24
        mistralai/mistral-7b-chat       0.00      0.00      0.00        53
      mistralai/mixtral-8x7b-chat       0.00      0.00      0.00        91
          zero-one-ai/Yi-34B-Chat       0.00      0.00      0.00        64

                         accuracy                   

In [ ]:
y_pred = router.predict(X_test)
test_indices = y_test.index if hasattr(y_test, "index") else None

# Recover the original rows used in the test split
_, test_df = train_test_split( df,test_size=0.2,
    random_state=42,
    stratify=best_model_labels)

test_df = test_df.reset_index(drop=True)
test_quality = quality_matrix.loc[test_df.index]
test_cost = cost_matrix.loc[test_df.index]

selected_quality = []
selected_cost = []

for i, model in enumerate(y_pred):
    row_index = test_df.index[i]
    selected_quality.append(test_quality.loc[row_index, model])
    selected_cost.append(test_cost.loc[row_index, model])

avg_quality = np.mean(selected_quality)
avg_cost = np.mean(selected_cost)
best_fixed_quality = quality_matrix.loc[test_df.index].mean().max()
best_fixed_cost = cost_matrix.loc[test_df.index].mean().min()

print("Router average quality:", round(avg_quality, 4))
print("Router average cost per prompt:", f"${avg_cost:.6f}")
print("Router cost per 1,000 prompts:", f"${avg_cost * 1000:.4f}")
print("Best fixed-model average quality:", round(best_fixed_quality, 4))
print("Cheapest fixed-model cost per prompt:", f"${best_fixed_cost:.6f}")

Router average quality: 0.6447
Router average cost per prompt: $0.000405
Router cost per 1,000 prompts: $0.4054
Best fixed-model average quality: 0.7172
Cheapest fixed-model cost per prompt: $0.000069


In [ ]:
test_quality = quality_matrix.iloc[test_idx].reset_index(drop=True)
test_cost = cost_matrix.iloc[test_idx].reset_index(drop=True)

router_quality = np.array([test_quality.iloc[i][model] for i, model in enumerate(y_pred)])
router_cost = np.array([test_cost.iloc[i][model] for i, model in enumerate(y_pred)])

best_fixed_model = quality_matrix.mean().idxmax()
cheapest_fixed_model = cost_matrix.mean().idxmin()

results = pd.DataFrame({
    "Strategy": ["RouteIQ Router", f"Fixed: {best_fixed_model}", f"Fixed: {cheapest_fixed_model}"],
    "Average Quality": [
        router_quality.mean(),
        test_quality[best_fixed_model].mean(),
        test_quality[cheapest_fixed_model].mean()
    ],
    "Average Cost": [
        router_cost.mean(),
        test_cost[best_fixed_model].mean(),
        test_cost[cheapest_fixed_model].mean()
    ]
})

results["Cost per 1000 Prompts"] = results["Average Cost"] * 1000
print(results.round(6).to_string(index=False))

                        Strategy  Average Quality  Average Cost  Cost per 1000 Prompts
                  RouteIQ Router         0.622167      0.000253               0.252906
       Fixed: gpt-4-1106-preview         0.784411      0.003221               3.220694
Fixed: mistralai/mistral-7b-chat         0.306364      0.000045               0.045306


In [ ]:
gpt4_quality = results.loc[results["Strategy"].str.contains("gpt-4"), "Average Quality"].iloc[0]
gpt4_cost = results.loc[results["Strategy"].str.contains("gpt-4"), "Average Cost"].iloc[0]
mistral_quality = results.loc[results["Strategy"].str.contains("mistral-7b"), "Average Quality"].iloc[0]
mistral_cost = results.loc[results["Strategy"].str.contains("mistral-7b"), "Average Cost"].iloc[0]

route_quality = results.loc[results["Strategy"] == "RouteIQ Router", "Average Quality"].iloc[0]
route_cost = results.loc[results["Strategy"] == "RouteIQ Router", "Average Cost"].iloc[0]

print("Quality retained vs GPT-4:", f"{route_quality / gpt4_quality * 100:.2f}%")
print("Cost saved vs GPT-4:", f"{(1 - route_cost / gpt4_cost) * 100:.2f}%")
print("Quality gain vs Mistral 7B:", f"{route_quality - mistral_quality:.4f}")
print("Extra cost vs Mistral 7B per 1,000 prompts:", f"${(route_cost - mistral_cost) * 1000:.4f}")

Quality retained vs GPT-4: 79.32%
Cost saved vs GPT-4: 92.15%
Quality gain vs Mistral 7B: 0.3158
Extra cost vs Mistral 7B per 1,000 prompts: $0.2076


In [ ]:
quality_values = quality_matrix.to_numpy(dtype=float)
cost_values = cost_matrix.to_numpy(dtype=float)

normalized_costs = cost_values / cost_values.max()

lambda_value = 0.2
utility_scores = quality_values - lambda_value * normalized_costs

utility_labels = np.array(models)[utility_scores.argmax(axis=1)]

print("Quality-cost-aware labels created!")
print("Number of labels:", len(utility_labels))
print("\nModel distribution:")
print(pd.Series(utility_labels).value_counts().to_string())

Quality-cost-aware labels created!
Number of labels: 36178

Model distribution:
mistralai/mistral-7b-chat            11166
WizardLM/WizardLM-13B-V1.2            8353
mistralai/mixtral-8x7b-chat           6129
zero-one-ai/Yi-34B-Chat               5283
claude-instant-v1                     1573
gpt-3.5-turbo-1106                    1050
gpt-4-1106-preview                     731
meta/llama-2-70b-chat                  666
meta/code-llama-instruct-34b-chat      475
claude-v1                              414
claude-v2                              338


In [ ]:
indices = np.arange(len(df))

train_idx, test_idx = train_test_split( indices,
    test_size=0.3,
    random_state=42,
    stratify=utility_labels)

X_train, X_test = prompt_embeddings[train_idx], prompt_embeddings[test_idx]
y_train, y_test = utility_labels[train_idx], utility_labels[test_idx]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("Embedding dimensions:", X_train.shape[1])

Training samples: 25324
Testing samples: 10854
Embedding dimensions: 384


In [ ]:
classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weights = dict(zip(classes, weights))

router = RandomForestClassifier(n_estimators=200, class_weight=class_weights, random_state=42, n_jobs=-1)
router.fit(X_train, y_train)

y_pred = router.predict(X_test)

print("Test accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

Test accuracy: 0.3299

Classification report:
                                   precision    recall  f1-score   support

       WizardLM/WizardLM-13B-V1.2       0.29      0.22      0.25      2506
                claude-instant-v1       0.30      0.01      0.01       472
                        claude-v1       0.57      0.03      0.06       124
                        claude-v2       0.00      0.00      0.00       101
               gpt-3.5-turbo-1106       0.44      0.04      0.07       315
               gpt-4-1106-preview       0.26      0.02      0.04       219
meta/code-llama-instruct-34b-chat       0.00      0.00      0.00       143
            meta/llama-2-70b-chat       0.00      0.00      0.00       200
        mistralai/mistral-7b-chat       0.32      0.70      0.44      3350
      mistralai/mixtral-8x7b-chat       0.33      0.04      0.07      1839
          zero-one-ai/Yi-34B-Chat       0.42      0.36      0.39      1585

                         accuracy                   

In [ ]:
test_quality = quality_matrix.iloc[test_idx].reset_index(drop=True)
test_cost = cost_matrix.iloc[test_idx].reset_index(drop=True)

router_quality = np.array([test_quality.iloc[i][model] for i, model in enumerate(y_pred)])
router_cost = np.array([test_cost.iloc[i][model] for i, model in enumerate(y_pred)])

best_fixed_model = quality_matrix.mean().idxmax()
cheapest_fixed_model = cost_matrix.mean().idxmin()

results = pd.DataFrame({
    "Strategy": ["RouteIQ Cost-Aware Router", f"Fixed: {best_fixed_model}", f"Fixed: {cheapest_fixed_model}"],
    "Average Quality": [
        router_quality.mean(),
        test_quality[best_fixed_model].mean(),
        test_quality[cheapest_fixed_model].mean()
    ],
    "Average Cost": [
        router_cost.mean(),
        test_cost[best_fixed_model].mean(),
        test_cost[cheapest_fixed_model].mean()
]})

results["Cost per 1000 Prompts"] = results["Average Cost"] * 1000
print(results.round(6).to_string(index=False))

                        Strategy  Average Quality  Average Cost  Cost per 1000 Prompts
       RouteIQ Cost-Aware Router         0.415635      0.000076               0.076019
       Fixed: gpt-4-1106-preview         0.782559      0.003293               3.292543
Fixed: mistralai/mistral-7b-chat         0.305952      0.000046               0.045635


In [ ]:
test_quality = quality_matrix.iloc[test_idx].reset_index(drop=True)
test_cost = cost_matrix.iloc[test_idx].reset_index(drop=True)

model_comparison = pd.DataFrame({
    "Average Quality": test_quality.mean(),
    "Average Cost": test_cost.mean()
})

model_comparison["Cost per 1000 Prompts"] = model_comparison["Average Cost"] * 1000
model_comparison["Quality per Dollar"] = model_comparison["Average Quality"] / model_comparison["Average Cost"]

print(model_comparison.sort_values("Average Cost").round(6).to_string())

                                   Average Quality  Average Cost  Cost per 1000 Prompts  Quality per Dollar
mistralai/mistral-7b-chat                 0.305952      0.000046               0.045635         6704.344620
WizardLM/WizardLM-13B-V1.2                0.432315      0.000073               0.072670         5949.012198
mistralai/mixtral-8x7b-chat               0.547987      0.000135               0.135005         4059.014426
meta/code-llama-instruct-34b-chat         0.200820      0.000172               0.172123         1166.724092
zero-one-ai/Yi-34B-Chat                   0.649502      0.000186               0.185647         3498.585420
meta/llama-2-70b-chat                     0.328782      0.000202               0.202012         1627.538205
claude-instant-v1                         0.602478      0.000233               0.232609         2590.095871
gpt-3.5-turbo-1106                        0.623392      0.000243               0.243458         2560.569582
claude-v1                   

In [ ]:
train_idx_new, val_idx = train_test_split(
    train_idx,
    test_size=0.2,
    random_state=42,
    stratify=utility_labels[train_idx]
)

X_train = prompt_embeddings[train_idx_new]
X_val = prompt_embeddings[val_idx]

y_train = utility_labels[train_idx_new]
y_val = utility_labels[val_idx]

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Test samples:", len(test_idx))

Training samples: 20259
Validation samples: 5065
Test samples: 10854


In [ ]:
classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weights = dict(zip(classes, weights))

router = RandomForestClassifier(n_estimators=200, class_weight=class_weights, random_state=42, n_jobs=-1)
router.fit(X_train, y_train)

val_pred = router.predict(X_val)

print("Validation accuracy:", round(accuracy_score(y_val, val_pred), 4))
print("Validation samples:", len(y_val))

Validation accuracy: 0.3346
Validation samples: 5065


In [ ]:
val_quality = quality_matrix.iloc[val_idx].reset_index(drop=True)
val_cost = cost_matrix.iloc[val_idx].reset_index(drop=True)

val_router_quality = np.array([val_quality.iloc[i][model] for i, model in enumerate(val_pred)])
val_router_cost = np.array([val_cost.iloc[i][model] for i, model in enumerate(val_pred)])

best_fixed_model = val_quality.mean().idxmax()
cheapest_fixed_model = val_cost.mean().idxmin()

validation_results = pd.DataFrame({
    "Strategy": ["RouteIQ Cost-Aware", f"Fixed: {best_fixed_model}", f"Fixed: {cheapest_fixed_model}"],
    "Average Quality": [val_router_quality.mean(), val_quality[best_fixed_model].mean(), val_quality[cheapest_fixed_model].mean()],
    "Average Cost": [val_router_cost.mean(), val_cost[best_fixed_model].mean(), val_cost[cheapest_fixed_model].mean()]
})

validation_results["Cost per 1000 Prompts"] = validation_results["Average Cost"] * 1000
print(validation_results.round(6).to_string(index=False))

                        Strategy  Average Quality  Average Cost  Cost per 1000 Prompts
              RouteIQ Cost-Aware         0.417345      0.000075               0.074791
       Fixed: gpt-4-1106-preview         0.790661      0.003275               3.274727
Fixed: mistralai/mistral-7b-chat         0.306288      0.000046               0.045854


In [ ]:
lambda_values = [0.0, 0.05, 0.1, 0.2, 0.3, 0.5, 0.75, 1.0]
quality_values = quality_matrix.to_numpy(dtype=float)
cost_values = cost_matrix.to_numpy(dtype=float)
normalized_costs = cost_values / cost_values.max()

for value in lambda_values:
    utility_scores = quality_values - value * normalized_costs
    labels = np.array(models)[utility_scores.argmax(axis=1)]
    print(f"Lambda={value:<4} | Unique models selected: {len(np.unique(labels))}")

Lambda=0.0  | Unique models selected: 11
Lambda=0.05 | Unique models selected: 11
Lambda=0.1  | Unique models selected: 11
Lambda=0.2  | Unique models selected: 11
Lambda=0.3  | Unique models selected: 11
Lambda=0.5  | Unique models selected: 11
Lambda=0.75 | Unique models selected: 11
Lambda=1.0  | Unique models selected: 11


In [ ]:
val_quality = quality_matrix.iloc[val_idx].to_numpy(dtype=float)
val_cost = cost_matrix.iloc[val_idx].to_numpy(dtype=float)
normalized_val_cost = val_cost / cost_values.max()

lambda_results = []

for value in lambda_values:
    utility = val_quality - value * normalized_val_cost
    selected = utility.argmax(axis=1)

    rows = np.arange(len(selected))
    avg_quality = val_quality[rows, selected].mean()
    avg_cost = val_cost[rows, selected].mean()

    lambda_results.append({
        "Lambda": value,
        "Average Quality": avg_quality,
        "Average Cost": avg_cost,
        "Cost per 1000 Prompts": avg_cost * 1000
    })

lambda_results = pd.DataFrame(lambda_results)
print(lambda_results.round(6).to_string(index=False))

 Lambda  Average Quality  Average Cost  Cost per 1000 Prompts
   0.00          0.91227      0.000580               0.580407
   0.05          0.91227      0.000227               0.226670
   0.10          0.91227      0.000227               0.226670
   0.20          0.91227      0.000227               0.226670
   0.30          0.91227      0.000227               0.226670
   0.50          0.91227      0.000227               0.226670
   0.75          0.91227      0.000227               0.226670
   1.00          0.91227      0.000227               0.226670


In [ ]:
for value in lambda_values:
    utility = val_quality - value * normalized_val_cost
    selected = utility.argmax(axis=1)
    counts = pd.Series(np.array(models)[selected]).value_counts()
    print(f"\nLambda = {value}")
    print(counts.to_string())


Lambda = 0.0
gpt-3.5-turbo-1106                   3315
claude-instant-v1                     780
gpt-4-1106-preview                    319
claude-v1                             294
claude-v2                             150
mistralai/mixtral-8x7b-chat            57
WizardLM/WizardLM-13B-V1.2             53
zero-one-ai/Yi-34B-Chat                38
mistralai/mistral-7b-chat              31
meta/llama-2-70b-chat                  17
meta/code-llama-instruct-34b-chat      11

Lambda = 0.05
mistralai/mistral-7b-chat            1563
WizardLM/WizardLM-13B-V1.2           1170
mistralai/mixtral-8x7b-chat           858
zero-one-ai/Yi-34B-Chat               740
claude-instant-v1                     220
gpt-3.5-turbo-1106                    147
gpt-4-1106-preview                    103
meta/llama-2-70b-chat                  93
meta/code-llama-instruct-34b-chat      66
claude-v1                              58
claude-v2                              47

Lambda = 0.1
mistralai/mistral-7b-chat        

In [ ]:
previous_selection = None

for value in lambda_values:
    utility = val_quality - value * normalized_val_cost
    selected = utility.argmax(axis=1)

    if previous_selection is None:
        changed = len(selected)
    else:
        changed = np.sum(selected != previous_selection)

    print(f"Lambda={value:<4} | Prompts with changed selection: {changed}")
    previous_selection = selected

Lambda=0.0  | Prompts with changed selection: 5065
Lambda=0.05 | Prompts with changed selection: 4451
Lambda=0.1  | Prompts with changed selection: 0
Lambda=0.2  | Prompts with changed selection: 0
Lambda=0.3  | Prompts with changed selection: 0
Lambda=0.5  | Prompts with changed selection: 0
Lambda=0.75 | Prompts with changed selection: 0
Lambda=1.0  | Prompts with changed selection: 0


In [ ]:
lambda_values_fine = [0.05, 0.06, 0.07, 0.08, 0.09, 0.1]
lambda_results_fine = []

for value in lambda_values_fine:
    utility = val_quality - value * normalized_val_cost
    selected = utility.argmax(axis=1)
    rows = np.arange(len(selected))

    avg_quality = val_quality[rows, selected].mean()
    avg_cost = val_cost[rows, selected].mean()

    lambda_results_fine.append({
        "Lambda": value,
        "Average Quality": avg_quality,
        "Average Cost": avg_cost,
        "Cost per 1000 Prompts": avg_cost * 1000
    })

print(pd.DataFrame(lambda_results_fine).round(6).to_string(index=False))

 Lambda  Average Quality  Average Cost  Cost per 1000 Prompts
   0.05          0.91227      0.000227                0.22667
   0.06          0.91227      0.000227                0.22667
   0.07          0.91227      0.000227                0.22667
   0.08          0.91227      0.000227                0.22667
   0.09          0.91227      0.000227                0.22667
   0.10          0.91227      0.000227                0.22667


In [ ]:
lambda_value = 0.05

train_quality = quality_matrix.iloc[train_idx_new].to_numpy(dtype=float)
train_cost = cost_matrix.iloc[train_idx_new].to_numpy(dtype=float)
normalized_train_cost = train_cost / cost_values.max()

train_utility = train_quality - lambda_value * normalized_train_cost
y_train_cost = np.array(models)[train_utility.argmax(axis=1)]

print("Training prompts:", len(y_train_cost))
print("Models selected:", len(np.unique(y_train_cost)))
print("\nLabel distribution:")
print(pd.Series(y_train_cost).value_counts().to_string())

Training prompts: 20259
Models selected: 11

Label distribution:
mistralai/mistral-7b-chat            6253
WizardLM/WizardLM-13B-V1.2           4677
mistralai/mixtral-8x7b-chat          3431
zero-one-ai/Yi-34B-Chat              2958
claude-instant-v1                     881
gpt-3.5-turbo-1106                    588
gpt-4-1106-preview                    410
meta/llama-2-70b-chat                 373
meta/code-llama-instruct-34b-chat     266
claude-v1                             232
claude-v2                             190


In [ ]:

classes = np.unique(y_train_cost)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train_cost)
class_weights_cost = dict(zip(classes, weights))

router_cost_aware = RandomForestClassifier(n_estimators=200, class_weight=class_weights_cost, random_state=42, n_jobs=-1)
router_cost_aware.fit(X_train, y_train_cost)

val_pred_cost = router_cost_aware.predict(X_val)

print("Validation accuracy:", round(accuracy_score(y_val, val_pred_cost), 4))
print("Validation predictions:", len(val_pred_cost))

Validation accuracy: 0.3335
Validation predictions: 5065


In [ ]:
val_quality_df = quality_matrix.iloc[val_idx].reset_index(drop=True)
val_cost_df = cost_matrix.iloc[val_idx].reset_index(drop=True)

router_quality = np.array([val_quality_df.iloc[i][model] for i, model in enumerate(val_pred_cost)])
router_cost = np.array([val_cost_df.iloc[i][model] for i, model in enumerate(val_pred_cost)])

gpt4_quality = val_quality_df["gpt-4-1106-preview"].mean()
gpt4_cost = val_cost_df["gpt-4-1106-preview"].mean()

mistral_quality = val_quality_df["mistralai/mistral-7b-chat"].mean()
mistral_cost = val_cost_df["mistralai/mistral-7b-chat"].mean()

print("Router quality:", round(router_quality.mean(), 4))
print("Router cost per 1,000 prompts: $", round(router_cost.mean() * 1000, 4))
print("GPT-4 quality:", round(gpt4_quality, 4))
print("GPT-4 cost per 1,000 prompts: $", round(gpt4_cost * 1000, 4))
print("Mistral 7B quality:", round(mistral_quality, 4))
print("Mistral 7B cost per 1,000 prompts: $", round(mistral_cost * 1000, 4))
print("Cost savings vs GPT-4 (%):", round((1 - router_cost.mean() / gpt4_cost) * 100, 2))

Router quality: 0.4158
Router cost per 1,000 prompts: $ 0.0753
GPT-4 quality: 0.7907
GPT-4 cost per 1,000 prompts: $ 3.2747
Mistral 7B quality: 0.3063
Mistral 7B cost per 1,000 prompts: $ 0.0459
Cost savings vs GPT-4 (%): 97.7


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
import pandas as pd
import numpy as np

lambda_values = [0.0, 0.01, 0.025, 0.05, 0.075, 0.1]
results = []

train_quality = quality_matrix.iloc[train_idx_new].to_numpy(dtype=float)
train_cost = cost_matrix.iloc[train_idx_new].to_numpy(dtype=float)
val_quality = quality_matrix.iloc[val_idx].to_numpy(dtype=float)
val_cost = cost_matrix.iloc[val_idx].to_numpy(dtype=float)

for lam in lambda_values:
    labels = np.array(models)[(train_quality - lam * train_cost / cost_values.max()).argmax(axis=1)]
    classes = np.unique(labels)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=labels)

    router = RandomForestClassifier(n_estimators=150, class_weight=dict(zip(classes, weights)), random_state=42, n_jobs=-1)
    router.fit(X_train, labels)
    predictions = router.predict(X_val)

    selected_indices = np.array([models.index(model) for model in predictions])
    rows = np.arange(len(predictions))
    results.append({"lambda": lam, "quality": val_quality[rows, selected_indices].mean(), "cost_per_1000": val_cost[rows, selected_indices].mean() * 1000})

print(pd.DataFrame(results).round(4).to_string(index=False))

 lambda  quality  cost_per_1000
  0.000   0.6281         0.2538
  0.010   0.4252         0.0753
  0.025   0.4252         0.0753
  0.050   0.4252         0.0753
  0.075   0.4252         0.0753
  0.100   0.4252         0.0753


In [ ]:
for lam in [0.0, 0.01]:
    utility = train_quality - lam * train_cost / cost_values.max()
    labels = np.array(models)[utility.argmax(axis=1)]

    print(f"\nLambda = {lam}")
    print(pd.Series(labels).value_counts().to_string())


Lambda = 0.0
gpt-3.5-turbo-1106                   13092
claude-instant-v1                     3205
gpt-4-1106-preview                    1264
claude-v1                             1211
claude-v2                              613
mistralai/mixtral-8x7b-chat            270
zero-one-ai/Yi-34B-Chat                180
WizardLM/WizardLM-13B-V1.2             157
mistralai/mistral-7b-chat              152
meta/llama-2-70b-chat                   70
meta/code-llama-instruct-34b-chat       45

Lambda = 0.01
mistralai/mistral-7b-chat            6253
WizardLM/WizardLM-13B-V1.2           4677
mistralai/mixtral-8x7b-chat          3431
zero-one-ai/Yi-34B-Chat              2958
claude-instant-v1                     881
gpt-3.5-turbo-1106                    588
gpt-4-1106-preview                    410
meta/llama-2-70b-chat                 373
meta/code-llama-instruct-34b-chat     266
claude-v1                             232
claude-v2                             190


In [ ]:
print("Quality range:", round(np.nanmin(quality_values), 4), "to", round(np.nanmax(quality_values), 4))
print("Cost range:", round(np.nanmin(cost_values), 6), "to", round(np.nanmax(cost_values), 6))
print("Median positive cost:", round(np.median(cost_values[cost_values > 0]), 6))
print("\nMean quality by model:")
print(quality_matrix.mean().sort_values(ascending=False).round(4).to_string())
print("\nMean cost per 1,000 prompts by model:")
print((cost_matrix.mean() * 1000).sort_values().round(4).to_string())

Quality range: 0.0 to 1.0
Cost range: 4e-06 to 0.05073
Median positive cost: 0.000194

Mean quality by model:
gpt-4-1106-preview                   0.7862
zero-one-ai/Yi-34B-Chat              0.6522
claude-v2                            0.6404
claude-v1                            0.6346
gpt-3.5-turbo-1106                   0.6237
claude-instant-v1                    0.6031
mistralai/mixtral-8x7b-chat          0.5513
WizardLM/WizardLM-13B-V1.2           0.4345
meta/llama-2-70b-chat                0.3315
mistralai/mistral-7b-chat            0.3083
meta/code-llama-instruct-34b-chat    0.2039

Mean cost per 1,000 prompts by model:
mistralai/mistral-7b-chat            0.0458
WizardLM/WizardLM-13B-V1.2           0.0730
mistralai/mixtral-8x7b-chat          0.1348
meta/code-llama-instruct-34b-chat    0.1726
zero-one-ai/Yi-34B-Chat              0.1858
meta/llama-2-70b-chat                0.2030
claude-instant-v1                    0.2339
gpt-3.5-turbo-1106                   0.2440
claude-v1      

In [ ]:
mean_cost = np.mean(cost_values)
median_cost = np.median(cost_values[cost_values > 0])

print("Mean cost per prompt:", round(mean_cost, 6))
print("Median positive cost:", round(median_cost, 6))
print("Mean cost per 1,000 prompts:", round(mean_cost * 1000, 4))

Mean cost per prompt: 0.000831
Median positive cost: 0.000194
Mean cost per 1,000 prompts: 0.8308


In [ ]:
lambda_values = [0.0, 0.0001, 0.00025, 0.0005, 0.001]
mean_cost = cost_values.mean()

for lam in lambda_values:
    utility = train_quality - lam * (train_cost / mean_cost)
    labels = np.array(models)[utility.argmax(axis=1)]

    print(f"\nLambda = {lam}")
    print(pd.Series(labels).value_counts().to_string())


Lambda = 0.0
gpt-3.5-turbo-1106                   13092
claude-instant-v1                     3205
gpt-4-1106-preview                    1264
claude-v1                             1211
claude-v2                              613
mistralai/mixtral-8x7b-chat            270
zero-one-ai/Yi-34B-Chat                180
WizardLM/WizardLM-13B-V1.2             157
mistralai/mistral-7b-chat              152
meta/llama-2-70b-chat                   70
meta/code-llama-instruct-34b-chat       45

Lambda = 0.0001
mistralai/mistral-7b-chat            6253
WizardLM/WizardLM-13B-V1.2           4677
mistralai/mixtral-8x7b-chat          3431
zero-one-ai/Yi-34B-Chat              2958
claude-instant-v1                     881
gpt-3.5-turbo-1106                    588
gpt-4-1106-preview                    410
meta/llama-2-70b-chat                 373
meta/code-llama-instruct-34b-chat     266
claude-v1                             232
claude-v2                             190

Lambda = 0.00025
mistralai/mistra

In [ ]:
labels_0 = np.array(models)[train_quality.argmax(axis=1)]

labels_001 = np.array(models)[
    (train_quality - 0.0001 * (train_cost / mean_cost)).argmax(axis=1)
]

print("Total training prompts:", len(labels_0))
print("Prompts changing model:", np.sum(labels_0 != labels_001))
print("Percentage changing:", round(np.mean(labels_0 != labels_001) * 100, 2), "%")

Total training prompts: 20259
Prompts changing model: 17700
Percentage changing: 87.37 %


In [ ]:
best_quality_idx = train_quality.argmax(axis=1)
cost_aware_idx = (train_quality - 0.0001 * (train_cost / mean_cost)).argmax(axis=1)

rows = np.arange(len(train_quality))
quality_drop = train_quality[rows, best_quality_idx] - train_quality[rows, cost_aware_idx]
cost_saved = train_cost[rows, best_quality_idx] - train_cost[rows, cost_aware_idx]

print("Prompts changing models:", np.sum(best_quality_idx != cost_aware_idx))
print("Average quality sacrificed:", round(quality_drop.mean(), 4))
print("Median quality sacrificed:", round(np.median(quality_drop), 4))
print("Average cost saved per prompt: $", round(cost_saved.mean(), 6))
print("Prompts with no quality loss:", np.sum(quality_drop == 0))
print("Prompts with quality loss:", np.sum(quality_drop > 0))

Prompts changing models: 17700
Average quality sacrificed: 0.0
Median quality sacrificed: 0.0
Average cost saved per prompt: $ 0.000362
Prompts with no quality loss: 20259
Prompts with quality loss: 0


In [ ]:
changed = best_quality_idx != cost_aware_idx
rows = np.arange(len(train_quality))

print("Changed prompts:", changed.sum())
print("Changed prompts with equal quality:", np.sum(changed & (train_quality[rows, best_quality_idx] == train_quality[rows, cost_aware_idx])))
print("Changed prompts with lower quality:", np.sum(changed & (train_quality[rows, best_quality_idx] > train_quality[rows, cost_aware_idx])))
print("Maximum quality difference:", np.max(train_quality[rows, best_quality_idx] - train_quality[rows, cost_aware_idx]))
print("Average cost before: $", round(train_cost[rows, best_quality_idx].mean(), 6))
print("Average cost after: $", round(train_cost[rows, cost_aware_idx].mean(), 6))

Changed prompts: 17700
Changed prompts with equal quality: 17700
Changed prompts with lower quality: 0
Maximum quality difference: 0.0
Average cost before: $ 0.000599
Average cost after: $ 0.000237


In [ ]:
val_utility = val_quality - 0.0001 * (val_cost / mean_cost)
oracle_idx = val_utility.argmax(axis=1)
rows = np.arange(len(val_quality))

oracle_quality = val_quality[rows, oracle_idx].mean()
oracle_cost = val_cost[rows, oracle_idx].mean()

quality_only_idx = val_quality.argmax(axis=1)
quality_only_cost = val_cost[rows, quality_only_idx].mean()

print("Cost-aware policy quality:", round(oracle_quality, 4))
print("Cost-aware policy cost / 1,000:", round(oracle_cost * 1000, 4))
print("Quality-only policy quality:", round(val_quality[rows, quality_only_idx].mean(), 4))
print("Quality-only policy cost / 1,000:", round(quality_only_cost * 1000, 4))
print("Cost reduction (%):", round((1 - oracle_cost / quality_only_cost) * 100, 2))

Cost-aware policy quality: 0.9123
Cost-aware policy cost / 1,000: 0.2267
Quality-only policy quality: 0.9123
Quality-only policy cost / 1,000: 0.5804
Cost reduction (%): 60.95


RouteIQ: Cost-Aware LLM Routing and Quality Optimization

RouteIQ is a cost-aware LLM routing platform designed to select the most suitable language model for each input prompt while balancing response quality and inference cost. The system uses prompt embeddings generated with sentence-transformers/all-MiniLM-L6-v2 (384-dimensional vectors) as input features and learns routing decisions from the recorded quality and cost scores of 11 candidate language models in the RouterBench dataset.

Our initial target was to achieve at least 0.65 average quality while substantially reducing costs compared with always selecting GPT-4.

Cost-Aware Utility Function

To generate cost-aware routing labels, we use the following utility function:

Utility(i, j) = Quality(i, j) − λ × (Cost(i, j) / Mean Cost)

Where:
- Quality(i, j): Recorded quality score of model j for prompt i.
- Cost(i, j): Recorded cost of model j for prompt i.
- Mean Cost: Average cost across all model-prompt pairs in the dataset.
- λ (lambda): Controls the trade-off between quality and cost.

When λ = 0, the selection prioritizes quality alone. As λ increases, expensive models receive a larger penalty, encouraging the routing policy to select cheaper alternatives. This may reduce inference costs but can also sacrifice quality.

Preliminary Validation Results

RouteIQ achieved an average recorded quality score of 0.9123 at an estimated cost of approximately $0.58 per 1,000 prompts under the ideal quality-only policy on the validation dataset. This result exceeds our initial quality target of 0.65. The result is an ideal-policy benchmark, not the final performance of the learned routing model on unseen test data.

These results exceed our initial quality target of 0.65. However, they represent ideal per-prompt selections based on known recorded quality and cost scores, not the performance of the learned router on unseen prompts.

The next objective is to train and evaluate the router on validation data, measure how closely it approaches the ideal policy, and assess its final quality-cost trade-off on an untouched test set.

In [ ]:
y_train_quality = np.array(models)[train_quality.argmax(axis=1)]
y_val_quality = np.array(models)[val_quality.argmax(axis=1)]

router_quality_only = RandomForestClassifier(
    n_estimators=200, class_weight="balanced",
    random_state=42, n_jobs=-1
)
router_quality_only.fit(X_train, y_train_quality)

val_pred_quality = router_quality_only.predict(X_val)
rows = np.arange(len(val_pred_quality))
selected_idx = np.array([models.index(model) for model in val_pred_quality])

print("Validation routing accuracy:", round(accuracy_score(y_val_quality, val_pred_quality), 4))
print("Router quality:", round(val_quality[rows, selected_idx].mean(), 4))
print("Router cost / 1,000 prompts: $", round(val_cost[rows, selected_idx].mean() * 1000, 4))

Validation routing accuracy: 0.6519
Router quality: 0.6283
Router cost / 1,000 prompts: $ 0.2536


In [ ]:
oracle_idx = val_quality.argmax(axis=1)
router_idx = np.array([models.index(model) for model in val_pred_quality])
rows = np.arange(len(val_quality))

oracle_quality = val_quality[rows, oracle_idx].mean()
router_quality = val_quality[rows, router_idx].mean()
oracle_cost = val_cost[rows, oracle_idx].mean()
router_cost = val_cost[rows, router_idx].mean()
gpt4_cost = val_cost[:, models.index("gpt-4-1106-preview")].mean()

print("Oracle quality:", round(oracle_quality, 4))
print("Router quality:", round(router_quality, 4))
print("Quality gap:", round(oracle_quality - router_quality, 4))
print("Router cost / 1,000: $", round(router_cost * 1000, 4))
print("Cost savings vs GPT-4 (%):", round((1 - router_cost / gpt4_cost) * 100, 2))

Oracle quality: 0.9123
Router quality: 0.6283
Quality gap: 0.284
Router cost / 1,000: $ 0.2536
Cost savings vs GPT-4 (%): 92.26


In [ ]:
print("Predicted model distribution:")
print(pd.Series(val_pred_quality).value_counts().to_string())

print("\nActual best-model distribution:")
print(pd.Series(y_val_quality).value_counts().to_string())

Predicted model distribution:
gpt-3.5-turbo-1106            5035
claude-instant-v1               11
gpt-4-1106-preview               9
claude-v1                        6
claude-v2                        2
WizardLM/WizardLM-13B-V1.2       1
zero-one-ai/Yi-34B-Chat          1

Actual best-model distribution:
gpt-3.5-turbo-1106                   3315
claude-instant-v1                     780
gpt-4-1106-preview                    319
claude-v1                             294
claude-v2                             150
mistralai/mixtral-8x7b-chat            57
WizardLM/WizardLM-13B-V1.2             53
zero-one-ai/Yi-34B-Chat                38
mistralai/mistral-7b-chat              31
meta/llama-2-70b-chat                  17
meta/code-llama-instruct-34b-chat      11


In [ ]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train_quality)
y_val_encoded = label_encoder.transform(y_val_quality)

router_xgb = XGBClassifier(
    objective="multi:softprob",
    num_class=len(label_encoder.classes_),
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

router_xgb.fit(X_train, y_train_encoded)
val_pred_encoded = router_xgb.predict(X_val).astype(int)
val_pred_xgb = label_encoder.inverse_transform(val_pred_encoded)

rows = np.arange(len(val_pred_xgb))
selected_idx = np.array([models.index(model) for model in val_pred_xgb])

print("Validation accuracy:", round(accuracy_score(y_val_quality, val_pred_xgb), 4))
print("Validation quality:", round(val_quality[rows, selected_idx].mean(), 4))
print("Cost / 1,000 prompts: $", round(val_cost[rows, selected_idx].mean() * 1000, 4))
print("\nPredicted model distribution:")
print(pd.Series(val_pred_xgb).value_counts().to_string())

Validation accuracy: 0.6505
Validation quality: 0.6297
Cost / 1,000 prompts: $ 0.2577

Predicted model distribution:
gpt-3.5-turbo-1106            5001
gpt-4-1106-preview              28
claude-instant-v1               26
claude-v1                        6
claude-v2                        2
WizardLM/WizardLM-13B-V1.2       1
zero-one-ai/Yi-34B-Chat          1


In [ ]:
best_quality = train_quality.max(axis=1)
tie_counts = np.isclose(train_quality, best_quality[:, None], atol=1e-8).sum(axis=1)

print("Prompts with one best model:", np.sum(tie_counts == 1))
print("Prompts with multiple equally best models:", np.sum(tie_counts > 1))
print("Maximum number of tied models:", tie_counts.max())
print("Average tied models per prompt:", round(tie_counts.mean(), 3))

Prompts with one best model: 1391
Prompts with multiple equally best models: 18868
Maximum number of tied models: 11
Average tied models per prompt: 6.017


In [ ]:
quality_max = train_quality.max(axis=1, keepdims=True)
best_mask = np.isclose(train_quality, quality_max, atol=1e-8)
cost_for_selection = np.where(best_mask, train_cost, np.inf)

best_indices_cost_tiebreak = cost_for_selection.argmin(axis=1)
y_train_tiebreak = np.array(models)[best_indices_cost_tiebreak]

print("New training label distribution:")
print(pd.Series(y_train_tiebreak).value_counts().to_string())

print("\nAverage label quality:", round(train_quality[np.arange(len(train_quality)), best_indices_cost_tiebreak].mean(), 4))
print("Average label cost / 1,000 prompts: $", round(train_cost[np.arange(len(train_cost)), best_indices_cost_tiebreak].mean() * 1000, 4))

New training label distribution:
mistralai/mistral-7b-chat            6253
WizardLM/WizardLM-13B-V1.2           4677
mistralai/mixtral-8x7b-chat          3431
zero-one-ai/Yi-34B-Chat              2958
claude-instant-v1                     881
gpt-3.5-turbo-1106                    588
gpt-4-1106-preview                    410
meta/llama-2-70b-chat                 373
meta/code-llama-instruct-34b-chat     266
claude-v1                             232
claude-v2                             190

Average label quality: 0.9176
Average label cost / 1,000 prompts: $ 0.2374


The new cost-tiebreak label-generation strategy improves RouteIQ by selecting the cheapest model among those tied for the highest recorded quality score for each prompt. The new training labels achieve an average recorded quality score of 0.9176 at an estimated cost of $0.2374 per 1,000 prompts. Previously, the quality-only approach selected the first model with the highest score, resulting in validation oracle quality of 0.9123 at approximately $0.5804 per 1,000 prompts. Although these figures come from different data splits and are not a direct comparison, the new strategy provides a promising direction for reducing routing costs while preserving recorded quality. The next step is to train the XGBoost router using the new labels and evaluate its actual quality and cost on the validation set.

In [ ]:

tie_encoder = LabelEncoder()
y_train_tiebreak_encoded = tie_encoder.fit_transform(y_train_tiebreak)

router_xgb_tiebreak = XGBClassifier(
    objective="multi:softprob",
    num_class=len(tie_encoder.classes_),
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

router_xgb_tiebreak.fit(X_train, y_train_tiebreak_encoded)
val_pred_tiebreak = tie_encoder.inverse_transform(
    router_xgb_tiebreak.predict(X_val).astype(int)
)

rows = np.arange(len(val_pred_tiebreak))
selected_idx = np.array([models.index(model) for model in val_pred_tiebreak])

print("Validation quality:", round(val_quality[rows, selected_idx].mean(), 4))
print("Cost / 1,000 prompts: $", round(val_cost[rows, selected_idx].mean() * 1000, 4))
print("\nPredicted model distribution:")
print(pd.Series(val_pred_tiebreak).value_counts().to_string())

Validation quality: 0.4635
Cost / 1,000 prompts: $ 0.0827

Predicted model distribution:
mistralai/mistral-7b-chat            2762
WizardLM/WizardLM-13B-V1.2           1316
zero-one-ai/Yi-34B-Chat               747
mistralai/mixtral-8x7b-chat           207
gpt-4-1106-preview                     14
gpt-3.5-turbo-1106                      9
claude-instant-v1                       4
meta/code-llama-instruct-34b-chat       3
claude-v1                               2
meta/llama-2-70b-chat                   1


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_val_tiebreak = np.array(models)[
    np.where(
        np.isclose(val_quality, val_quality.max(axis=1, keepdims=True), atol=1e-8),
        val_cost,
        np.inf
    ).argmin(axis=1)
]

print("Routing accuracy:", round(accuracy_score(y_val_tiebreak, val_pred_tiebreak), 4))
print("\nClassification report:")
print(classification_report(y_val_tiebreak, val_pred_tiebreak, zero_division=0))

Routing accuracy: 0.351

Classification report:
                                   precision    recall  f1-score   support

       WizardLM/WizardLM-13B-V1.2       0.32      0.36      0.34      1170
                claude-instant-v1       0.00      0.00      0.00       220
                        claude-v1       0.50      0.02      0.03        58
                        claude-v2       0.00      0.00      0.00        47
               gpt-3.5-turbo-1106       0.33      0.02      0.04       147
               gpt-4-1106-preview       0.36      0.05      0.09       103
meta/code-llama-instruct-34b-chat       0.00      0.00      0.00        66
            meta/llama-2-70b-chat       0.00      0.00      0.00        93
        mistralai/mistral-7b-chat       0.34      0.60      0.44      1563
      mistralai/mixtral-8x7b-chat       0.32      0.08      0.13       858
          zero-one-ai/Yi-34B-Chat       0.45      0.46      0.45       740

                         accuracy                 

In [ ]:
print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("Training labels:", len(y_train_tiebreak))
print("Validation labels:", len(y_val_tiebreak))

print("Training embeddings contain NaN:", np.isnan(X_train).any())
print("Validation embeddings contain NaN:", np.isnan(X_val).any())

print("Unique training labels:", len(np.unique(y_train_tiebreak)))
print("Unique validation labels:", len(np.unique(y_val_tiebreak)))

X_train shape: (20259, 384)
X_val shape: (5065, 384)
Training labels: 20259
Validation labels: 5065
Training embeddings contain NaN: False
Validation embeddings contain NaN: False
Unique training labels: 11
Unique validation labels: 11


In [ ]:


train_quality = quality_matrix.iloc[train_idx_new].to_numpy(dtype=np.float32)
val_quality = quality_matrix.iloc[val_idx].to_numpy(dtype=np.float32)

y_train_multi = np.isclose(
    train_quality,
    train_quality.max(axis=1, keepdims=True),
    atol=1e-8
).astype(np.float32)

y_val_multi = np.isclose(
    val_quality,
    val_quality.max(axis=1, keepdims=True),
    atol=1e-8
).astype(np.float32)

print("X_train:", X_train.shape)
print("Training targets:", y_train_multi.shape)
print("X_val:", X_val.shape)
print("Validation targets:", y_val_multi.shape)
print("Average tied best models per prompt:", round(y_train_multi.sum(axis=1).mean(), 3))

X_train: (20259, 384)
Training targets: (20259, 11)
X_val: (5065, 384)
Validation targets: (5065, 11)
Average tied best models per prompt: 6.017


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_train_t = torch.tensor(y_train_multi, dtype=torch.float32)
y_val_t = torch.tensor(y_val_multi, dtype=torch.float32)

train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=256,
    shuffle=True
)

class RouteIQNet(nn.Module):
    def __init__(self, input_dim=384, num_models=11):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_models)
        )

    def forward(self, x):
        return self.network(x)

router_nn = RouteIQNet().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(router_nn.parameters(), lr=0.001)

best_val_loss = float("inf")
best_state = None
patience = 5
epochs_without_improvement = 0

for epoch in range(40):
    router_nn.train()
    train_loss = 0.0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = router_nn(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(xb)

    router_nn.eval()
    with torch.no_grad():
        val_logits = router_nn(X_val_t.to(device))
        val_loss = criterion(val_logits, y_val_t.to(device)).item()

    train_loss /= len(X_train_t)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {
            key: value.detach().cpu().clone()
            for key, value in router_nn.state_dict().items()
        }
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    print(f"Epoch {epoch + 1:02d} | Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f}")

    if epochs_without_improvement >= patience:
        print("Early stopping triggered.")
        break

router_nn.load_state_dict(best_state)
router_nn.eval()

print(f"\nDevice: {device}")
print(f"Best validation loss: {best_val_loss:.4f}")

Epoch 01 | Train loss: 0.5997 | Val loss: 0.5774
Epoch 02 | Train loss: 0.5749 | Val loss: 0.5717
Epoch 03 | Train loss: 0.5662 | Val loss: 0.5698
Epoch 04 | Train loss: 0.5597 | Val loss: 0.5738
Epoch 05 | Train loss: 0.5519 | Val loss: 0.5728
Epoch 06 | Train loss: 0.5448 | Val loss: 0.5751
Epoch 07 | Train loss: 0.5369 | Val loss: 0.5814
Epoch 08 | Train loss: 0.5305 | Val loss: 0.5805
Early stopping triggered.

Device: cuda
Best validation loss: 0.5698


In [ ]:
router_nn.eval()

with torch.no_grad():
    val_logits = router_nn(X_val_t.to(device))
    val_probs = torch.sigmoid(val_logits).cpu().numpy()

# Predict all models considered suitable
predicted_suitable = val_probs >= 0.5

# If no model crosses the threshold, select the highest-probability model
no_selection = ~predicted_suitable.any(axis=1)
predicted_suitable[no_selection, val_probs[no_selection].argmax(axis=1)] = True

# Choose the cheapest predicted-suitable model
val_cost = cost_matrix.iloc[val_idx].to_numpy(dtype=float)
val_quality = quality_matrix.iloc[val_idx].to_numpy(dtype=float)

candidate_costs = np.where(predicted_suitable, val_cost, np.inf)
selected_idx = candidate_costs.argmin(axis=1)
rows = np.arange(len(selected_idx))

selected_quality = val_quality[rows, selected_idx]
selected_cost = val_cost[rows, selected_idx]

print("Validation quality:", round(selected_quality.mean(), 4))
print("Cost / 1,000 prompts: $", round(selected_cost.mean() * 1000, 4))
print("Average suitable models predicted:", round(predicted_suitable.sum(axis=1).mean(), 3))
print("\nSelected model distribution:")
print(pd.Series(np.array(models)[selected_idx]).value_counts().to_string())

Validation quality: 0.5832
Cost / 1,000 prompts: $ 0.1715
Average suitable models predicted: 7.071

Selected model distribution:
mistralai/mixtral-8x7b-chat          1578
WizardLM/WizardLM-13B-V1.2           1533
zero-one-ai/Yi-34B-Chat              1516
meta/llama-2-70b-chat                 150
mistralai/mistral-7b-chat              82
gpt-3.5-turbo-1106                     52
claude-instant-v1                      52
claude-v1                              40
gpt-4-1106-preview                     40
claude-v2                              17
meta/code-llama-instruct-34b-chat       5


In [ ]:

import copy
import torch
import torch.nn as nn

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

class ResidualBlock(nn.Module):
    def __init__(self, hidden_dim=256, dropout=0.2):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim)
        )
        self.activation = nn.GELU()

    def forward(self, x):
        return self.activation(x + self.block(x))

class RouteIQNet(nn.Module):
    def __init__(self, input_dim=384, hidden_dim=256, num_models=11):
        super().__init__()
        self.input_layer = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU()
        )
        self.residual_blocks = nn.Sequential(
            ResidualBlock(hidden_dim),
            ResidualBlock(hidden_dim),
            ResidualBlock(hidden_dim)
        )
        self.output_layer = nn.Linear(hidden_dim, num_models)

    def forward(self, x):
        x = self.input_layer(x)
        x = self.residual_blocks(x)
        return self.output_layer(x)

router_nn = RouteIQNet().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(
    router_nn.parameters(), lr=0.0005, weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2
)

best_val_loss = float("inf")
best_state = None
patience = 6
epochs_without_improvement = 0

for epoch in range(40):
    router_nn.train()
    train_loss = 0.0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(router_nn(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(router_nn.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item() * len(xb)

    router_nn.eval()
    with torch.no_grad():
        val_loss = criterion(
            router_nn(X_val_t.to(device)),
            y_val_t.to(device)
        ).item()

    train_loss /= len(X_train_t)
    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(router_nn.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    print(
        f"Epoch {epoch + 1:02d} | "
        f"Train: {train_loss:.4f} | Val: {val_loss:.4f}"
    )

    if epochs_without_improvement >= patience:
        print("Early stopping triggered.")
        break

router_nn.load_state_dict(best_state)
router_nn.eval()
print(f"\nDevice: {device} | Best validation loss: {best_val_loss:.4f}")


Epoch 01 | Train: 0.5988 | Val: 0.5765
Epoch 02 | Train: 0.5745 | Val: 0.5734
Epoch 03 | Train: 0.5647 | Val: 0.5715
Epoch 04 | Train: 0.5536 | Val: 0.5740
Epoch 05 | Train: 0.5427 | Val: 0.5840
Epoch 06 | Train: 0.5276 | Val: 0.5827
Epoch 07 | Train: 0.5021 | Val: 0.6036
Epoch 08 | Train: 0.4874 | Val: 0.6174
Epoch 09 | Train: 0.4743 | Val: 0.6382
Early stopping triggered.

Device: cuda | Best validation loss: 0.5715


In [ ]:


router_nn.eval()

with torch.no_grad():
    val_logits = router_nn(X_val_t.to(device))
    val_probs = torch.sigmoid(val_logits).cpu().numpy()

val_cost = cost_matrix.iloc[val_idx].to_numpy(dtype=float)
val_quality = quality_matrix.iloc[val_idx].to_numpy(dtype=float)

results = []

for threshold in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    suitable = val_probs >= threshold

    empty = ~suitable.any(axis=1)
    suitable[empty, val_probs[empty].argmax(axis=1)] = True

    selected_idx = np.where(suitable, val_cost, np.inf).argmin(axis=1)
    rows = np.arange(len(selected_idx))

    selected_quality = val_quality[rows, selected_idx]
    selected_cost = val_cost[rows, selected_idx]

    results.append({
        "threshold": threshold,
        "quality": selected_quality.mean(),
        "cost_per_1000": selected_cost.mean() * 1000,
        "avg_candidates": suitable.sum(axis=1).mean()
    })

results_df = pd.DataFrame(results)
print(results_df.round(4).to_string(index=False))


 threshold  quality  cost_per_1000  avg_candidates
       0.3   0.3900         0.0690          9.2736
       0.4   0.5022         0.0989          8.2450
       0.5   0.5811         0.1573          6.7743
       0.6   0.6521         0.3281          5.1115
       0.7   0.7117         1.1188          3.2430
       0.8   0.7603         1.9915          2.0586
       0.9   0.7827         2.2676          1.2677


In [ ]:

threshold = 0.6
suitable = val_probs >= threshold

empty = ~suitable.any(axis=1)
suitable[empty, val_probs[empty].argmax(axis=1)] = True

val_cost = cost_matrix.iloc[val_idx].to_numpy(dtype=float)
val_quality = quality_matrix.iloc[val_idx].to_numpy(dtype=float)

selected_idx = np.where(suitable, val_cost, np.inf).argmin(axis=1)
rows = np.arange(len(selected_idx))

routing_df = pd.DataFrame({
    "model": np.array(models)[selected_idx],
    "quality": val_quality[rows, selected_idx],
    "cost": val_cost[rows, selected_idx]
})

summary = routing_df.groupby("model").agg(
    prompts=("model", "size"),
    avg_quality=("quality", "mean"),
    avg_cost=("cost", "mean")
).sort_values("prompts", ascending=False)

summary["prompt_share_pct"] = summary["prompts"] / len(routing_df) * 100

print(summary.round(4).to_string())
print("\nOverall quality:", round(routing_df["quality"].mean(), 4))
print("Cost per 1,000 prompts:", round(routing_df["cost"].mean() * 1000, 4))


                                   prompts  avg_quality  avg_cost  prompt_share_pct
model                                                                              
zero-one-ai/Yi-34B-Chat               1395       0.7484    0.0002           27.5420
gpt-3.5-turbo-1106                     954       0.5788    0.0004           18.8351
WizardLM/WizardLM-13B-V1.2             938       0.6009    0.0000           18.5192
mistralai/mixtral-8x7b-chat            768       0.6607    0.0001           15.1629
claude-instant-v1                      451       0.6242    0.0004            8.9042
gpt-4-1106-preview                     439       0.6478    0.0016            8.6673
mistralai/mistral-7b-chat               94       0.6383    0.0000            1.8559
claude-v1                                9       0.5556    0.0017            0.1777
meta/code-llama-instruct-34b-chat        9       0.1111    0.0002            0.1777
meta/llama-2-70b-chat                    5       0.4000    0.0003           

In [ ]:
al_cost = cost_matrix.iloc[val_idx].to_numpy(dtype=float)
val_quality = quality_matrix.iloc[val_idx].to_numpy(dtype=float)

results = []

for alpha in [0.0, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]:
    # alpha controls how strongly we favor lower-cost models.
    normalized_cost = val_cost / (val_cost.max() + 1e-12)
    routing_score = val_probs - alpha * normalized_cost

    selected_idx = routing_score.argmax(axis=1)
    rows = np.arange(len(selected_idx))

    selected_quality = val_quality[rows, selected_idx]
    selected_cost = val_cost[rows, selected_idx]

    results.append({
        "alpha": alpha,
        "quality": selected_quality.mean(),
        "cost_per_1000": selected_cost.mean() * 1000,
        "avg_candidates": 1.0
    })

policy_df = pd.DataFrame(results)
print(policy_df.round(4).to_string(index=False))


 alpha  quality  cost_per_1000  avg_candidates
   0.0   0.7890         2.3648             1.0
   0.1   0.7849         2.1577             1.0
   0.2   0.7823         1.7611             1.0
   0.3   0.7785         1.3155             1.0
   0.5   0.7749         0.9781             1.0
   0.7   0.7726         0.9092             1.0
   1.0   0.7728         0.8562             1.0


In [ ]:


torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_train_t = torch.tensor(y_train_multi, dtype=torch.float32)
y_val_t = torch.tensor(y_val_multi, dtype=torch.float32)

train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=256,
    shuffle=True
)

class RouteIQTransformer(nn.Module):
    def __init__(self, input_dim=384, d_model=384, nhead=12, num_models=11):
        super().__init__()

        self.projection = nn.Sequential(
            nn.Linear(input_dim, d_model),
            nn.LayerNorm(d_model)
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=768,
            dropout=0.2,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=3,
            enable_nested_tensor=False
        )

        self.output_norm = nn.LayerNorm(d_model)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 192),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(192, num_models)
        )

    def forward(self, x):
        x = self.projection(x).unsqueeze(1)
        x = self.encoder(x)
        x = x.squeeze(1)
        return self.classifier(self.output_norm(x))

router_transformer = RouteIQTransformer().to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(
    router_transformer.parameters(),
    lr=0.0003,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2
)

best_val_loss = float("inf")
best_state = None
patience = 6
epochs_without_improvement = 0

for epoch in range(40):
    router_transformer.train()
    train_loss = 0.0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()

        logits = router_transformer(xb)
        loss = criterion(logits, yb)
        loss.backward()

        nn.utils.clip_grad_norm_(router_transformer.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item() * len(xb)

    router_transformer.eval()
    with torch.no_grad():
        val_loss = criterion(
            router_transformer(X_val_t.to(device)),
            y_val_t.to(device)
        ).item()

    train_loss /= len(X_train_t)
    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(router_transformer.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    print(
        f"Epoch {epoch + 1:02d} | "
        f"Train loss: {train_loss:.4f} | "
        f"Val loss: {val_loss:.4f}"
    )

    if epochs_without_improvement >= patience:
        print("Early stopping triggered.")
        break

router_transformer.load_state_dict(best_state)
router_transformer.eval()

print(f"\nDevice: {device}")
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Parameters: {sum(p.numel() for p in router_transformer.parameters()):,}")


Epoch 01 | Train loss: 0.5971 | Val loss: 0.5764
Epoch 02 | Train loss: 0.5798 | Val loss: 0.5740
Epoch 03 | Train loss: 0.5742 | Val loss: 0.5737
Epoch 04 | Train loss: 0.5698 | Val loss: 0.5698
Epoch 05 | Train loss: 0.5661 | Val loss: 0.5709
Epoch 06 | Train loss: 0.5630 | Val loss: 0.5689
Epoch 07 | Train loss: 0.5598 | Val loss: 0.5735
Epoch 08 | Train loss: 0.5560 | Val loss: 0.5720
Epoch 09 | Train loss: 0.5514 | Val loss: 0.5740
Epoch 10 | Train loss: 0.5406 | Val loss: 0.5802
Epoch 11 | Train loss: 0.5354 | Val loss: 0.5830
Epoch 12 | Train loss: 0.5299 | Val loss: 0.5856
Early stopping triggered.

Device: cuda
Best validation loss: 0.5689
Parameters: 3,777,035


In [ ]:
router_transformer.eval()

with torch.no_grad():
    val_logits = router_transformer(X_val_t.to(device))
    val_probs = torch.sigmoid(val_logits).cpu().numpy()

val_cost = cost_matrix.iloc[val_idx].to_numpy(dtype=float)
val_quality = quality_matrix.iloc[val_idx].to_numpy(dtype=float)

results = []

for alpha in [0.0, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]:
    normalized_cost = val_cost / (val_cost.max() + 1e-12)
    routing_score = val_probs - alpha * normalized_cost

    selected_idx = routing_score.argmax(axis=1)
    rows = np.arange(len(selected_idx))

    results.append({
        "alpha": alpha,
        "quality": val_quality[rows, selected_idx].mean(),
        "cost_per_1000": val_cost[rows, selected_idx].mean() * 1000
    })

results_df = pd.DataFrame(results)
print(results_df.round(4).to_string(index=False))


 alpha  quality  cost_per_1000
   0.0   0.7885         2.2818
   0.1   0.7883         2.2077
   0.2   0.7870         2.1323
   0.3   0.7871         2.0342
   0.5   0.7832         1.4113
   0.7   0.7786         1.0410
   1.0   0.7752         0.9430


In [ ]:

mlp_results = [
    {"architecture": "Residual MLP", "alpha": 0.0, "quality": 0.7890, "cost": 2.3648},
    {"architecture": "Residual MLP", "alpha": 0.1, "quality": 0.7849, "cost": 2.1577},
    {"architecture": "Residual MLP", "alpha": 0.2, "quality": 0.7823, "cost": 1.7611},
    {"architecture": "Residual MLP", "alpha": 0.3, "quality": 0.7785, "cost": 1.3155},
    {"architecture": "Residual MLP", "alpha": 0.5, "quality": 0.7749, "cost": 0.9781},
    {"architecture": "Residual MLP", "alpha": 0.7, "quality": 0.7726, "cost": 0.9092},
    {"architecture": "Residual MLP", "alpha": 1.0, "quality": 0.7728, "cost": 0.8562}
]

transformer_results = [
    {"architecture": "Transformer", "alpha": 0.0, "quality": 0.7885, "cost": 2.2818},
    {"architecture": "Transformer", "alpha": 0.1, "quality": 0.7883, "cost": 2.2077},
    {"architecture": "Transformer", "alpha": 0.2, "quality": 0.7870, "cost": 2.1323},
    {"architecture": "Transformer", "alpha": 0.3, "quality": 0.7871, "cost": 2.0342},
    {"architecture": "Transformer", "alpha": 0.5, "quality": 0.7832, "cost": 1.4113},
    {"architecture": "Transformer", "alpha": 0.7, "quality": 0.7786, "cost": 1.0410},
    {"architecture": "Transformer", "alpha": 1.0, "quality": 0.7752, "cost": 0.9430}
]

comparison = pd.DataFrame(mlp_results + transformer_results)
pareto_mask = []

for i, row in comparison.iterrows():
    dominated = (
        (comparison["quality"] >= row["quality"]) &
        (comparison["cost"] <= row["cost"]) &
        ((comparison["quality"] > row["quality"]) |
         (comparison["cost"] < row["cost"]))
    ).any()
    pareto_mask.append(not dominated)

comparison["pareto_optimal"] = pareto_mask

print(
    comparison[comparison["pareto_optimal"]]
    .sort_values("cost")
    .round(4)
    .to_string(index=False)
)


architecture  alpha  quality   cost  pareto_optimal
Residual MLP    1.0   0.7728 0.8562            True
 Transformer    1.0   0.7752 0.9430            True
 Transformer    0.7   0.7786 1.0410            True
 Transformer    0.5   0.7832 1.4113            True
 Transformer    0.3   0.7871 2.0342            True
 Transformer    0.1   0.7883 2.2077            True
 Transformer    0.0   0.7885 2.2818            True
Residual MLP    0.0   0.7890 2.3648            True


In [ ]:

import json
import os
import torch

ARTIFACT_DIR = "/content/routeiq_artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

# Save model weights
torch.save(
    router_nn.state_dict(),
    os.path.join(ARTIFACT_DIR, "residual_mlp.pt")
)

# Save architecture configuration
model_config = {
    "architecture": "RouteIQNet",
    "input_dim": 384,
    "hidden_dim": 256,
    "num_models": len(models),
    "residual_blocks": 3,
    "activation": "GELU",
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "output_type": "multilabel_logits",
    "loss": "BCEWithLogitsLoss"
}

with open(os.path.join(ARTIFACT_DIR, "model_config.json"), "w") as f:
    json.dump(model_config, f, indent=4)

# Save candidate model order
with open(os.path.join(ARTIFACT_DIR, "candidate_models.json"), "w") as f:
    json.dump(list(models), f, indent=4)

# Save routing policy
routing_config = {
    "policy": "probability_minus_normalized_cost",
    "alpha": 0.7,
    "threshold": None,
    "cost_normalization": "validation_cost_matrix_global_max"
}

with open(os.path.join(ARTIFACT_DIR, "routing_config.json"), "w") as f:
    json.dump(routing_config, f, indent=4)

print("Saved artifacts:")
for filename in sorted(os.listdir(ARTIFACT_DIR)):
    print(filename)


Saved artifacts:
candidate_models.json
model_config.json
residual_mlp.pt
routing_config.json


In [ ]:

import os
import json
import shutil
import torch
from google.colab import files

artifact_dir = "/content/routeiq_artifacts"
os.makedirs(artifact_dir, exist_ok=True)

# Save trained residual MLP weights
torch.save(router_nn.state_dict(), f"{artifact_dir}/residual_mlp.pt")

# Save model configuration
config = {
    "architecture": "RouteIQNet",
    "input_dim": 384,
    "hidden_dim": 256,
    "num_models": len(models),
    "residual_blocks": 3,
    "activation": "GELU",
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2"
}

with open(f"{artifact_dir}/model_config.json", "w") as f:
    json.dump(config, f, indent=4)

# Save model names in their exact output-column order
with open(f"{artifact_dir}/candidate_models.json", "w") as f:
    json.dump(list(models), f, indent=4)

# Save routing policy
with open(f"{artifact_dir}/routing_config.json", "w") as f:
    json.dump({
        "policy": "probability_minus_normalized_cost",
        "alpha": 0.7
    }, f, indent=4)

# Create ZIP and download it
zip_path = shutil.make_archive(
    "/content/routeiq_residual_mlp_deployment",
    "zip",
    artifact_dir
)

print("ZIP created successfully:", zip_path)
files.download(zip_path)


ZIP created successfully: /content/routeiq_residual_mlp_deployment.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:


val_quality = quality_matrix.iloc[val_idx].to_numpy(dtype=float)
val_cost = cost_matrix.iloc[val_idx].to_numpy(dtype=float)
rows = np.arange(len(val_idx))

def evaluate_router(name, probabilities, alpha):
    normalized_cost = val_cost / (val_cost.max() + 1e-12)
    selected = (probabilities - alpha * normalized_cost).argmax(axis=1)

    quality = val_quality[rows, selected].mean()
    cost = val_cost[rows, selected].mean()

    return {
        "Model": name,
        "Alpha": alpha,
        "Quality": quality,
        "Cost / 1000": cost * 1000
    }

# Get predictions from the residual MLP
router_nn.eval()
with torch.no_grad():
    mlp_probs = torch.sigmoid(
        router_nn(X_val_t.to(device))
    ).cpu().numpy()

# Get predictions from the Transformer
router_transformer.eval()
with torch.no_grad():
    transformer_probs = torch.sigmoid(
        router_transformer(X_val_t.to(device))
    ).cpu().numpy()

results = []

for alpha in [0.0, 0.5, 0.7, 1.0]:
    results.append(evaluate_router("Residual MLP", mlp_probs, alpha))
    results.append(evaluate_router("Transformer", transformer_probs, alpha))

# Fixed-model baselines
for i, model_name in enumerate(models):
    results.append({
        "Model": f"Fixed: {model_name}",
        "Alpha": None,
        "Quality": val_quality[:, i].mean(),
        "Cost / 1000": val_cost[:, i].mean() * 1000
    })

results_df = pd.DataFrame(results)

# Cost reduction relative to always choosing GPT-4
gpt4_idx = list(models).index("gpt-4-1106-preview")
gpt4_cost = val_cost[:, gpt4_idx].mean() * 1000

results_df["Cost Reduction %"] = (
    (gpt4_cost - results_df["Cost / 1000"]) / gpt4_cost * 100
)

print("GPT-4 baseline cost per 1,000:", round(gpt4_cost, 4))
print("\nRouter comparison:")
print(
    results_df[results_df["Alpha"].notna()]
    .sort_values("Cost / 1000")
    .round(4)
    .to_string(index=False)
)

print("\nBest fixed models by quality:")
print(
    results_df[results_df["Alpha"].isna()]
    .sort_values("Quality", ascending=False)
    .head(5)
    .round(4)
    .to_string(index=False)
)


GPT-4 baseline cost per 1,000: 3.2747

Router comparison:
       Model  Alpha  Quality  Cost / 1000  Cost Reduction %
Residual MLP    1.0   0.7728       0.8562           73.8545
Residual MLP    0.7   0.7726       0.9092           72.2371
 Transformer    1.0   0.7752       0.9430           71.2035
Residual MLP    0.5   0.7749       0.9781           70.1326
 Transformer    0.7   0.7786       1.0410           68.2112
 Transformer    0.5   0.7832       1.4113           56.9037
 Transformer    0.0   0.7885       2.2818           30.3214
Residual MLP    0.0   0.7890       2.3648           27.7875

Best fixed models by quality:
                         Model  Alpha  Quality  Cost / 1000  Cost Reduction %
     Fixed: gpt-4-1106-preview    NaN   0.7907       3.2747            0.0000
Fixed: zero-one-ai/Yi-34B-Chat    NaN   0.6574       0.1869           94.2931
              Fixed: claude-v2    NaN   0.6482       2.4334           25.6905
              Fixed: claude-v1    NaN   0.6417       2.1481

# RouteIQ: Cost-Aware LLM Routing with a Residual MLP

## 1. Project Overview

RouteIQ is a cost-aware Large Language Model (LLM) routing system that selects an appropriate LLM for an incoming prompt. Instead of sending every request to an expensive model such as GPT-4, RouteIQ attempts to balance response quality with inference cost by routing prompts to different candidate models.

The system uses semantic embeddings to represent prompts and a neural network to estimate which candidate models are suitable for each request. A cost-aware decision policy then selects the final model.

**Primary objectives:**
- Maintain high response quality.
- Reduce inference costs compared with always using GPT-4.
- Learn routing decisions from historical prompt, quality, and cost data.
- Support deployment as a reusable routing component.

## 2. Dataset and Candidate Models

The project uses the RouterBench dataset, with 36,178 English prompts retained after preprocessing.

Each prompt has recorded quality and cost information for 11 candidate models:

1. WizardLM-13B
2. Claude Instant
3. Claude v1
4. Claude v2
5. GPT-3.5 Turbo
6. GPT-4
7. Code Llama 34B
8. Llama 2 70B
9. Mistral 7B
10. Mixtral 8x7B
11. Yi 34B

For every prompt, the dataset provides the information needed to compare candidate models by recorded quality and cost.

## 3. Prompt Representation

RouteIQ uses the Sentence-BERT model `sentence-transformers/all-MiniLM-L6-v2` to transform each prompt into a 384-dimensional embedding.

These embeddings capture semantic information about the input prompt and serve as features for the routing network.

**Input shape:** 384 features per prompt.

Using precomputed embeddings separates text representation from routing-model training and avoids repeatedly encoding prompts during every training epoch.

## 4. Why a Residual MLP?

We experimented with conventional classifiers, a multilabel neural network, a deeper residual MLP, and a Transformer encoder.

The residual MLP was selected as the preferred starting architecture because it provides competitive routing quality with a comparatively simple deployment structure.

Its main components are:

- **Input layer:** Projects the 384-dimensional embedding into a hidden representation.
- **Residual blocks:** Transform the hidden representation while adding skip connections.
- **GELU activation:** Introduces nonlinear transformations.
- **Layer normalization:** Helps stabilize hidden activations.
- **Dropout:** Regularizes training and helps reduce overfitting.
- **Output layer:** Produces 11 logits, one for each candidate LLM.

Residual connections allow information to flow through shortcut paths, helping gradients propagate through deeper networks.

## 5. Multilabel Learning

Several candidate models can have the same maximum recorded quality for a prompt. Therefore, forcing every prompt to have exactly one training label can discard useful information and introduce arbitrary tie-breaking.

RouteIQ uses a multilabel target:

- Each candidate model receives a target of 1 if its recorded quality equals the maximum quality for that prompt.
- All other candidate models receive a target of 0.

The network produces 11 logits. Applying the sigmoid function converts these logits into independent model suitability scores.

The training objective is Binary Cross-Entropy with Logits (`BCEWithLogitsLoss`), which combines sigmoid and binary cross-entropy in a numerically stable loss function.

These sigmoid scores should not automatically be interpreted as calibrated probabilities of success.

## 6. Cost-Aware Routing Policy

After obtaining the suitability scores, RouteIQ incorporates model cost into the selection decision.

For prompt \(x\) and candidate model \(m\), the experimental routing score is:

\[
S(x,m)=p_m(x)-\alpha\frac{c(x,m)}{C_{\max}}
\]

Where:

- \(p_m(x)\) is the sigmoid output for candidate model \(m\).
- \(c(x,m)\) is the recorded cost associated with that prompt-model pair.
- \(C_{\max}\) is the maximum cost used for normalization.
- \(\alpha\) controls the strength of the cost penalty.

The router selects the candidate with the highest routing score.

A smaller \(\alpha\) prioritizes model suitability scores. A larger \(\alpha\) penalizes expensive choices more strongly.

The normalization reference used in deployment must match the reference used during evaluation.

## 7. Experimental Results

The following are validation results from the residual MLP experiments:

| Alpha | Validation quality | Cost per 1,000 prompts |
|---:|---:|---:|
| 0.0 | 0.7890 | $2.3648 |
| 0.1 | 0.7849 | $2.1577 |
| 0.2 | 0.7823 | $1.7611 |
| 0.3 | 0.7785 | $1.3155 |
| 0.5 | 0.7749 | $0.9781 |
| 0.7 | 0.7726 | $0.9092 |
| 1.0 | 0.7728 | $0.8562 |

The configuration with \(\alpha=0.7\) is a useful candidate because it achieves a validation quality of 0.7726 at a recorded cost of $0.9092 per 1,000 prompts.

The \(\alpha=1.0\) configuration has a slightly lower cost, at $0.8562 per 1,000 prompts, and a marginally higher quality of 0.7728. Therefore, the final configuration should be selected based on the verified results and the project's quality-cost requirements rather than assuming \(\alpha=0.7\) is universally optimal.

These figures are validation metrics, not final test-set results.

## 8. Comparison with the Transformer Encoder

We also tested a Transformer encoder with:

- Model dimension: 384
- Attention heads: 12
- Encoder layers: 3
- Feed-forward dimension: 768
- GELU activation
- Multilabel output layer

The Transformer achieved competitive validation quality. However, it received one pooled embedding token per prompt, so its self-attention could not learn interactions among multiple prompt tokens.

The residual MLP remains a practical starting point for the current feature representation. A token-level Transformer experiment could be explored separately.

## 9. Training and Evaluation Strategy

The dataset is separated into training, validation, and test subsets.

- **Training set:** Fits the routing network.
- **Validation set:** Compares architectures and routing parameters.
- **Test set:** Reserved for final evaluation after selecting the configuration.

The evaluation measures recorded quality and cost for the model selected for each validation prompt. This evaluates the routing policy's performance against observed outcomes in the dataset.

The final evaluation should also compare the router with fixed-model baselines, including always using GPT-4, and report the cost reduction alongside quality.

## 10. Deployment Artifacts

The deployment package should contain:

- Trained Residual MLP weights.
- Model architecture configuration.
- Ordered list of candidate model names.
- Routing-policy configuration.
- Cost normalization reference.
- Embedding model identifier.
- Dependency versions and model-loading code.

At inference time, the system encodes the incoming prompt, obtains the 11 model suitability scores, applies the cost-aware selection rule, and returns the selected candidate model to the calling application.

The deployment implementation must reproduce the same architecture, model ordering, cost normalization, and routing formula used during evaluation.

## 11. Limitations and Future Improvements

1. **Historical outcomes:** Recorded quality and cost may not perfectly represent current provider performance or pricing.
2. **Probability calibration:** Sigmoid outputs are suitability scores and may require calibration.
3. **Latency:** The current experiments focus on quality and cost; latency should also be measured.
4. **Provider availability:** A production router needs fallbacks when a selected model is unavailable.
5. **Generalization:** Performance should be verified on the untouched test set and, eventually, on real requests.
6. **Fine-tuning:** QLoRA fine-tuning of a separate LLM can be investigated to improve task performance, but it does not replace the routing network itself.

## Conclusion

RouteIQ combines semantic prompt embeddings, a residual multilabel neural network, and a cost-aware selection policy to choose among multiple LLMs. Its goal is to achieve a practical balance between response quality and inference expenditure.

The residual MLP is the preferred starting architecture for deployment experiments. Final claims about quality and savings should be made only after evaluating the frozen test set and verifying that deployment reproduces the experimental routing policy.

In [ ]:


router_nn.eval()

X_test = prompt_embeddings[test_idx]
X_test_t = torch.tensor(X_test, dtype=torch.float32)

with torch.no_grad():
    test_logits = router_nn(X_test_t.to(device))
    test_probs = torch.sigmoid(test_logits).cpu().numpy()

test_quality = quality_matrix.iloc[test_idx].to_numpy(dtype=float)
test_cost = cost_matrix.iloc[test_idx].to_numpy(dtype=float)

alpha = 0.7
normalized_cost = test_cost / (val_cost.max() + 1e-12)
routing_scores = test_probs - alpha * normalized_cost

selected_idx = routing_scores.argmax(axis=1)
rows = np.arange(len(selected_idx))

selected_quality = test_quality[rows, selected_idx]
selected_cost = test_cost[rows, selected_idx]

gpt4_idx = list(models).index("gpt-4-1106-preview")
gpt4_quality = test_quality[:, gpt4_idx]
gpt4_cost = test_cost[:, gpt4_idx]

router_quality = selected_quality.mean()
router_cost = selected_cost.mean()
baseline_quality = gpt4_quality.mean()
baseline_cost = gpt4_cost.mean()

cost_reduction = (
    (baseline_cost - router_cost) / baseline_cost * 100
    if baseline_cost > 0 else np.nan
)

print(f"Test prompts: {len(test_idx):,}")
print(f"Router quality: {router_quality:.4f}")
print(f"Router cost / 1,000 prompts: ${router_cost * 1000:.4f}")
print(f"GPT-4 quality: {baseline_quality:.4f}")
print(f"GPT-4 cost / 1,000 prompts: ${baseline_cost * 1000:.4f}")
print(f"Cost reduction vs GPT-4: {cost_reduction:.2f}%")
print(f"Quality difference vs GPT-4: {router_quality - baseline_quality:+.4f}")

print("\nSelected model distribution:")
print(pd.Series(np.array(models)[selected_idx]).value_counts().to_string())


Test prompts: 10,854
Router quality: 0.7697
Router cost / 1,000 prompts: $0.9455
GPT-4 quality: 0.7826
GPT-4 cost / 1,000 prompts: $3.2925
Cost reduction vs GPT-4: 71.28%
Quality difference vs GPT-4: -0.0128

Selected model distribution:
gpt-4-1106-preview                   6350
claude-instant-v1                    2128
zero-one-ai/Yi-34B-Chat              1716
gpt-3.5-turbo-1106                    462
mistralai/mixtral-8x7b-chat           140
meta/code-llama-instruct-34b-chat      34
claude-v1                              16
meta/llama-2-70b-chat                   7
claude-v2                               1
